# 02 — Preprocessing & Feature Engineering

Làm sạch data và chuẩn bị cho training.

**Thứ tự chạy:** 01_eda → **02_processing** → 03_modeling

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

from src.preprocessor import AutoPreprocessor

## 1. Load Data

In [2]:
df = pd.read_csv('../data/healthcare_dataset.csv')
df_processed = df.copy()

## 2. Xử lý Billing Amount có giá trị âm

In [3]:
print(f'   Trước: {(df_processed["Billing Amount"] < 0).sum()} giá trị âm')
df_processed['Billing Amount'] = df_processed['Billing Amount'].clip(lower=0)
print(f'   Sau:   {(df_processed["Billing Amount"] < 0).sum()} giá trị âm')

   Trước: 108 giá trị âm
   Sau:   0 giá trị âm


## 3. Drop các cột không có predictive value

In [4]:
col_to_drop = ['Name', 'Doctor', 'Hospital']
df_processed = df_processed.drop(columns=col_to_drop)
print(f'Dropped: {col_to_drop}')
print(f'Còn lại: {df_processed.shape[1]} cột')

Dropped: ['Name', 'Doctor', 'Hospital']
Còn lại: 12 cột


## 4. Label Encoding cho các cột Categorical

`LabelEncoder`: ánh xạ mỗi category thành một số nguyên

Ví dụ: `Male→1, Female→0` | `A+→0, A-→1, B+→2...`

In [5]:
le = LabelEncoder()
categorical_cols = [
    'Gender', 'Blood Type', 'Medical Condition',
    'Insurance Provider', 'Admission Type', 'Medication'
]

label_mapping = {}
for col in categorical_cols:
    original_values = df_processed[col].unique()
    df_processed[col] = le.fit_transform(df_processed[col])
    encoded_values = sorted(df_processed[col].unique())
    mapping = dict(zip(sorted(original_values), encoded_values))
    label_mapping[col] = mapping
    print(f'   {col:<20}: {list(original_values[:3])}... → {encoded_values[:3]}...')

   Gender              : ['Male', 'Female']... → [np.int64(0), np.int64(1)]...
   Blood Type          : ['B-', 'A+', 'A-']... → [np.int64(0), np.int64(1), np.int64(2)]...
   Medical Condition   : ['Cancer', 'Obesity', 'Diabetes']... → [np.int64(0), np.int64(1), np.int64(2)]...
   Insurance Provider  : ['Blue Cross', 'Medicare', 'Aetna']... → [np.int64(0), np.int64(1), np.int64(2)]...
   Admission Type      : ['Urgent', 'Emergency', 'Elective']... → [np.int64(0), np.int64(1), np.int64(2)]...
   Medication          : ['Paracetamol', 'Ibuprofen', 'Aspirin']... → [np.int64(0), np.int64(1), np.int64(2)]...


## 5. Encode Target Variable

In [6]:
le_target = LabelEncoder()
df_processed['Test Results'] = le_target.fit_transform(df_processed['Test Results'])
target_name = le_target.classes_.tolist()
print(f'   Classes: {list(zip(target_name, range(len(target_name))))}')
df_processed.head(3)

   Classes: [('Abnormal', 0), ('Inconclusive', 1), ('Normal', 2)]


,Age,Gender,Blood Type,Medical Condition,Date of Admission,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,30,1,5,2,2024-01-31,1,18856.281306,328,2,2024-02-02,3,2
1,62,1,0,5,2019-08-20,3,33643.327287,265,1,2019-08-26,1,1
2,76,0,1,5,2022-09-22,0,27955.096079,205,1,2022-10-07,0,2


## 6. Feature Engineering từ Datetime

In [7]:
df_processed['Date of Admission'] = pd.to_datetime(df_processed['Date of Admission'])
df_processed['Discharge Date']    = pd.to_datetime(df_processed['Discharge Date'])

In [8]:
# Feature 1: số ngày nằm viện
df_processed['Length of Stay'] = (
    df_processed['Discharge Date'] - df_processed['Date of Admission']
).dt.days
print(f' Length of Stay: min={df_processed["Length of Stay"].min()}, '
      f'max={df_processed["Length of Stay"].max()}, '
      f'mean={df_processed["Length of Stay"].mean():.1f} ngày')

 Length of Stay: min=1, max=30, mean=15.5 ngày


In [9]:
# Feature 2: Tháng nhập viện
df_processed['Admission Month'] = df_processed['Date of Admission'].dt.month
print(' Admission Month: 1-12 (January=1, December=12)')

# Feature 3: Ngày trong tuần
df_processed['Admission DayOfWeek'] = df_processed['Date of Admission'].dt.dayofweek
print(' Admission DayOfWeek: 0=Monday, 6=Sunday')

# Drop cột datetime gốc
df_processed.drop(columns=['Date of Admission', 'Discharge Date'], inplace=True)

print(f'\nDataset sau Feature Engineering:')
print(f'   Shape: {df_processed.shape}')
print(f'   Features: {list(df_processed.columns)}')

 Admission Month: 1-12 (January=1, December=12)
 Admission DayOfWeek: 0=Monday, 6=Sunday

Dataset sau Feature Engineering:
   Shape: (55500, 13)
   Features: ['Age', 'Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Medication', 'Test Results', 'Length of Stay', 'Admission Month', 'Admission DayOfWeek']


## 7. Train / Val / Test Split & Scaling

Chia data theo tỷ lệ **80% train — 10% val — 10% test**.

| Set | Tỷ lệ | Mục đích |
|-----|-------|----------|
| Train | 80% | Dạy model học |
| **Val** | **10%** | **Tune hyperparameter, early stopping** |
| Test | 10% | Đánh giá cuối cùng — chỉ dùng 1 lần |

⚠️ **Quy tắc vàng**: chỉ `fit()` scaler trên **train set**, dùng `transform()` cho val và test.

**Tại sao cần val set riêng?**
Nếu dùng test set để tune model → model biết thông tin test → đánh giá bị lạc quan (optimistic bias).
Val set giữ vai trò 'bài kiểm tra thử', test set là 'bài thi thật' chỉ chấm 1 lần.

In [10]:
from sklearn.model_selection import train_test_split

X = df_processed.drop(columns=['Test Results'])
y = df_processed['Test Results']

# Bước 1: tách test ra trước (10%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.1,        # 10% test
    random_state=42,
    stratify=y
)

# Bước 2: tách val từ phần còn lại (10% / 90% ≈ 11.1%)
# → kết quả: train=80%, val=10%, test=10%
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1/0.9,    # 10% val từ 90% còn lại
    random_state=42,
    stratify=y_temp
)

n = len(X)
print(f'Tổng: {n:,} samples')
print(f'X_train : {X_train.shape}  ({len(X_train)/n*100:.0f}%)')
print(f'X_val   : {X_val.shape}    ({len(X_val)/n*100:.0f}%)')
print(f'X_test  : {X_test.shape}   ({len(X_test)/n*100:.0f}%)')


Tổng: 55,500 samples
X_train : (44400, 12)  (80%)
X_val   : (5550, 12)    (10%)
X_test  : (5550, 12)   (10%)


In [11]:
scaler = StandardScaler()

# fit CHỈ trên train → tránh data leakage
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_val_scaled   = scaler.transform(X_val)         # chỉ transform
X_test_scaled  = scaler.transform(X_test)        # chỉ transform

print(f'X_train_scaled : {X_train_scaled.shape}')
print(f'X_val_scaled   : {X_val_scaled.shape}')
print(f'X_test_scaled  : {X_test_scaled.shape}')


X_train_scaled : (44400, 12)
X_val_scaled   : (5550, 12)
X_test_scaled  : (5550, 12)


## 8. Hoặc dùng AutoPreprocessor (1 lệnh làm tất cả)

Thay cho tất cả các bước thủ công ở trên, `AutoPreprocessor` tự động:
drop → clip âm → feature engineering → encode → **split 80/10/10** → scale

In [12]:
df_raw = pd.read_csv('../data/healthcare_dataset.csv')

prep = AutoPreprocessor(
    target_col   = 'Test Results',
    drop_cols    = ['Name', 'Doctor', 'Hospital'],
    val_size     = 0.1,   # 10% validation
    test_size    = 0.1,   # 10% test
    random_state = 42
)

# Trả về 6 giá trị khi val_size > 0
X_train_auto, X_val_auto, X_test_auto, \
y_train_auto, y_val_auto,  y_test_auto = prep.fit_transform(df_raw)


   🗑️  Dropped: ['Name', 'Doctor', 'Hospital']
   🔧 Clipped 108 giá trị âm → 0 trong 'Billing Amount'
✅ Preprocessing xong!
   Tổng: 55,500 samples
   X_train : (44400, 12)  (80%)
   X_val   : (5550, 12)   (10%)
   X_test  : (5550, 12)   (10%)
   Features: ['Age', 'Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Medication', 'Length of Stay', 'Admission Month', 'Admission DayOfWeek']
   Target classes: ['Abnormal', 'Inconclusive', 'Normal']


In [13]:
import numpy as np
np.save('../data/X_train.npy', X_train_scaled)
np.save('../data/X_val.npy',   X_val_scaled)
np.save('../data/X_test.npy',  X_test_scaled)
np.save('../data/y_train.npy', y_train.values)
np.save('../data/y_val.npy',   y_val.values)
np.save('../data/y_test.npy',  y_test.values)

import joblib
joblib.dump(scaler, '../data/scaler.pkl')
print('Đã lưu X_train/val/test .npy, y_train/val/test .npy, scaler.pkl vào data/')


Đã lưu X_train/val/test .npy, y_train/val/test .npy, scaler.pkl vào data/
